# Evaluasi RAGAS Chatbot Lydera

## Setup

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

AI_SERVICES = Path.cwd().parents[1]
sys.path.insert(0, str(AI_SERVICES))

from annotation import config
from chatbot import agent, chunk as chunk_mod, llm_ext

EVAL_DIR = AI_SERVICES / "chatbot" / "eval"


## Model Evaluator

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

ragas_llm = LangchainLLMWrapper(ChatOpenAI(
    model=config.CHAT_MODEL, base_url=config.AZURE_OPENAI_BASE_URL,
    api_key=config.AZURE_OPENAI_API_KEY, reasoning_effort="low",
), bypass_temperature=True, bypass_n=True)

ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(
    model=config.EMBEDDING_MODEL, base_url=config.EMBEDDING_BASE_URL, api_key=config.EMBEDDING_API_KEY,
))

## Index dari Data Anotasi

In [1]:
def load_chapter_chunks(path, chapter_id):
    blocks = json.loads(path.read_text(encoding="utf-8"))
    return chunk_mod.build_chunks(chapter_id, blocks)


chunks_bab1 = load_chapter_chunks(EVAL_DIR / "blocks" / "bab1.json", 1)
chunks_bab2 = load_chapter_chunks(EVAL_DIR / "blocks" / "bab2.json", 2)
all_chunks = chunks_bab1 + chunks_bab2

vectors = llm_ext.embed_texts([c.text for c in all_chunks])
index = [
    {"reference": str(c.block_ids[0]), "block_ids": c.block_ids, "heading": c.heading,
     "text": c.text, "chapter_id": c.chapter_id, "embedding": np.array(v)}
    for c, v in zip(all_chunks, vectors)
]

print(f"total chunk: {len(index)} (Bab 1: {len(chunks_bab1)}, Bab 2: {len(chunks_bab2)})")

total chunk: 116 (Bab 1: 68, Bab 2: 48)


## Simulasi Pencarian Modul

In [ ]:
def make_executor(index, top_k):
    def executor(query):
        qv = np.array(llm_ext.embed_text(query))
        scored = []
        for item in index:
            sim = float(np.dot(qv, item["embedding"]) / (np.linalg.norm(qv) * np.linalg.norm(item["embedding"])))
            scored.append((sim, item))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [
            {"reference": it["reference"], "block_ids": it["block_ids"], "heading": it["heading"],
             "text": it["text"], "chapter_id": it["chapter_id"], "similarity": round(sim, 4)}
            for sim, it in scored[:top_k]
        ]
    return executor


search_module_executor = make_executor(index, top_k=config.TOP_K_CHUNKS)

## Dataset Evaluasi

In [2]:
dataset = json.loads((EVAL_DIR / "dataset.json").read_text(encoding="utf-8"))
df_dataset = pd.DataFrame(dataset)
print(df_dataset[["id", "chapter_title", "category", "question"]].to_string(index=False))

     id                      chapter_title category                                                                                                     question
bab1-q1 Komposisi Fungsi dan Fungsi Invers   konsep                                                                      Apa syarat suatu relasi disebut fungsi?
bab1-q2 Komposisi Fungsi dan Fungsi Invers   konsep                                                      Apa itu relasi, dan bagaimana relasi direpresentasikan?
bab1-q3 Komposisi Fungsi dan Fungsi Invers   konsep                                                                    Apa definisi formal komposisi fungsi f∘g?
bab1-q4 Komposisi Fungsi dan Fungsi Invers   konsep                                                     Apa syarat agar dua fungsi f dan g dapat dikomposisikan?
bab1-q5 Komposisi Fungsi dan Fungsi Invers   konsep                                                      Apa perbedaan fungsi injektif, surjektif, dan bijektif?
bab1-q6 Komposisi Fungsi dan Fungs

17 pertanyaan: 9 dari Bab 1, 8 dari Bab 2, terbagi konsep (9), prosedur (4), campuran (2), overview (2). Pertanyaan overview menyebut nama bab secara eksplisit supaya tidak ambigu saat pencarian lintas dua bab digabung jadi satu index. Semua ground truth diambil verbatim/rangkum dari isi modul, bukan soal hitung bernomor.

## Top-K Hasil Pencarian per Pertanyaan

In [ ]:
sample = next(item for item in dataset if item["id"] == "bab1-q4")
hits = search_module_executor(sample["question"])

df_topk = pd.DataFrame([
    {"rank": r, "similarity": h["similarity"], "heading": h["heading"], "reference": h["reference"]}
    for r, h in enumerate(hits, start=1)
])
print(sample["question"])
print(df_topk.to_string(index=False))

Apa syarat agar dua fungsi f dan g dapat dikomposisikan?
 rank  similarity                   heading reference
    1      0.7721   Syarat Komposisi Fungsi       239
    2      0.7306                Eksplorasi       228
    3      0.7078               Ayo Mencoba       224
    4      0.6914 Definisi Komposisi Fungsi       220
    5      0.6880            Masalah Ketiga       249


## Menjalankan Chatbot

In [3]:
def retrieved_contexts_from(result):
    texts = {}
    for call in result["tool_calls"]:
        try:
            payload = json.loads(call["result"])
        except json.JSONDecodeError:
            continue
        if call["name"] == "search_module":
            for c in payload.get("chunks", []):
                texts[c["reference"]] = c["text"]
        elif call["name"] == "query_wolfram_alpha":
            if payload.get("answer"):
                texts[call["arguments"]] = payload["answer"]
        elif call["name"] in ("search_oer", "search_academic_web"):
            for item in payload.get("results", []):
                texts[item["url"]] = f"{item['title']} {item['snippet']}"
    return list(texts.values())


samples = []
for item in dataset:
    result = agent.run(item["question"], [], search_module_executor)
    contexts = retrieved_contexts_from(result)
    samples.append({**item, "status": result["status"], "response": result["message"], "retrieved_contexts": contexts})
    print(f"{item['id']}: status={result['status']} n_contexts={len(contexts)}")

bab1-q1: status=answered n_contexts=3
bab1-q2: status=answered n_contexts=2
bab1-q3: status=answered n_contexts=2
bab1-q4: status=answered n_contexts=2
bab1-q5: status=answered n_contexts=3
bab1-q6: status=answered n_contexts=2
bab1-q7: status=answered n_contexts=1
bab1-q8: status=answered n_contexts=3
bab1-q9: status=answered n_contexts=1
bab2-q1: status=answered n_contexts=4
bab2-q2: status=answered n_contexts=1
bab2-q3: status=answered n_contexts=7
bab2-q4: status=answered n_contexts=2
bab2-q5: status=answered n_contexts=7
bab2-q6: status=answered n_contexts=6
bab2-q7: status=answered n_contexts=2
bab2-q8: status=answered n_contexts=1


## Evaluasi RAGAS

In [ ]:
from ragas import evaluate
from ragas.dataset_schema import EvaluationDataset
from ragas.metrics import Faithfulness, LLMContextPrecisionWithReference, LLMContextRecall, ResponseRelevancy

rows = [
    {"user_input": s["question"], "retrieved_contexts": s["retrieved_contexts"] or [""],
     "response": s["response"], "reference": s["reference"]}
    for s in samples
]
eval_dataset = EvaluationDataset.from_list(rows)

metrics = [Faithfulness(), LLMContextPrecisionWithReference(), LLMContextRecall(), ResponseRelevancy()]
result = evaluate(dataset=eval_dataset, metrics=metrics, llm=ragas_llm, embeddings=ragas_embeddings, raise_exceptions=True)

df = result.to_pandas()
df.insert(0, "id", [s["id"] for s in samples])
df.insert(1, "chapter", [s["chapter"] for s in samples])
df.insert(2, "category", [s["category"] for s in samples])

In [4]:
df[['id', 'chapter', 'category', 'faithfulness', 'llm_context_precision_with_reference', 'context_recall', 'answer_relevancy']]

     id  chapter category  faithfulness  llm_context_precision_with_reference  context_recall  answer_relevancy
bab1-q1        1   konsep      1.000000                              1.000000             1.0          0.875391
bab1-q2        1   konsep      1.000000                              1.000000             1.0          0.978316
bab1-q3        1   konsep      1.000000                              1.000000             1.0          0.835567
bab1-q4        1   konsep      1.000000                              1.000000             1.0          0.921582
bab1-q5        1   konsep      1.000000                              1.000000             1.0          0.912855
bab1-q6        1 prosedur      0.666667                              1.000000             1.0          0.962883
bab1-q7        1 prosedur      1.000000                              1.000000             1.0          0.770788
bab1-q8        1 campuran      1.000000                              1.000000             1.0          0

In [5]:
df.groupby("chapter")[["faithfulness", "llm_context_precision_with_reference", "context_recall", "answer_relevancy"]].mean()

         faithfulness  llm_context_precision_with_reference  context_recall  answer_relevancy
chapter
1            0.962963                               1.000000          1.0000          0.861505
2            0.812500                               0.943899          0.9375          0.844244


In [6]:
df.groupby("category")[["faithfulness", "llm_context_precision_with_reference", "context_recall", "answer_relevancy"]].mean()

          faithfulness  llm_context_precision_with_reference  context_recall  answer_relevancy
category
campuran      1.000000                               0.908333        1.000000          0.823368
konsep        0.944444                               0.970503        0.944444          0.907972
overview      0.500000                               1.000000        1.000000          0.623083
prosedur      0.916667                               1.000000        1.000000          0.860711
